# Example 10: Weighted Area rHEALPix Remapping

Mass-conserving area-weighted methane remap on an rHEALPix DGGS mesh.


## Initialize Workflow Configuration

Load configuration from `workflow_config.yaml`.


In [ ]:
import os
import pyvista as pv

# Use static PyVista backend to render lightweight images and stay within MyBinder memory limits
pv.set_jupyter_backend("static")
from uraster import URaster, compile_remap_workflow, load_workflow_config

# Load Tier 1 configuration from workflow_config.yaml
config = load_workflow_config("workflow_config.yaml")


## Preprocess: Resample Input Raster

Downsample the high-resolution methane raster to 1.0 degree to demonstrate area-weighted remapping.


In [ ]:
from osgeo import osr
from pyearth.toolbox.management.raster.resample import resample_raster

# Retrieve the input raster path resolved from Pooch
sFilename_raster = config["rasters"]["source_rasters"][0]
sFilename_source_raster_resample = sFilename_raster.replace(".tif", "_resample.tif")

if not os.path.exists(sFilename_source_raster_resample):
    default_srs = osr.SpatialReference()
    default_srs.ImportFromEPSG(4326)
    pProjection_target = default_srs.ExportToWkt()
    resample_raster(
        sFilename_raster,
        sFilename_source_raster_resample,
        1.0,
        1.0,
        sResampleAlg="average",
        dMissing_value_source=0,
        pProjection_target_in=pProjection_target,
    )

# Point configuration to resampled raster and compile
config["rasters"]["source_rasters"] = [sFilename_source_raster_resample]
aConfig = compile_remap_workflow(config)
pRaster = URaster(aConfig)


## 2. Setup Case

Reads mesh topology and raster datasets, handling coordinate reference systems and date-line crossings.


In [ ]:
# Setup reads in mesh and raster datasets, validating topology and spatial extents
pRaster.setup()


## 3. Report Inputs

Inspect metadata and statistics for input raster(s) and mesh.


In [ ]:
# Report metadata and statistics for input raster(s) and mesh
pRaster.report_inputs()


## 4. Visualize Source Mesh

Render 3D source mesh topology. Camera focus and parameters are automatically handled.


In [ ]:
# Visualize source mesh topology
pRaster.visualize_source_mesh()


## 5. Run Remapping

Execute raster-to-mesh remapping using the method configured in `workflow_config.yaml`.


In [ ]:
# Execute remapping
pRaster.run_remap()


## 6. Report Outputs

Inspect summary statistics of the remapped variable on target mesh cells.


In [ ]:
# Report summary statistics of remapped output
pRaster.report_outputs()


## 7. Visualize Target Mesh

Render remapped zonal statistics on the 3D globe with colormap and focus handled automatically.


In [ ]:
# Visualize remapped variable on target mesh
pRaster.visualize_target_mesh()


## 8. Cleanup

Clean up temporary intermediate files.


In [ ]:
# Clean up temporary scratch files
pRaster.cleanup()
print("Done!")
